In [1]:
import torch
from transformers import AutoTokenizer, GPT2LMHeadModel, GPT2Config
from datasets import load_dataset

In [2]:
tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
tokenizer.pad_token = tokenizer.eos_token

config = GPT2Config(vocab_size=len(tokenizer))
model = GPT2LMHeadModel(config)

In [3]:
DATASET_NAME = "aslon1213/uzbek-language-corpus"
ds = load_dataset(DATASET_NAME)

In [5]:
def tokenize(examples):
    outputs = [text + tokenizer.eos_token for text in examples["text"]]
    return tokenizer(outputs)

train_ds = ds["train"].map(
    tokenize,
    batched=True,
    remove_columns=["text", "chars", "words"]
)

Map:   0%|          | 0/1212450 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1180 > 1024). Running this sequence through the model will result in indexing errors


In [6]:
block_size = 1024

def group_texts(examples):
    # Concatenate all the token arrays in the current batch
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    
    # Drop the small remainder of tokens at the very end of the dataset
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
        
    # Split the massive concatenated array into blocks of 'block_size'
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    
    # For training from scratch (causal modeling), the target labels are the exact same as the inputs.
    # The model handles shifting the tokens by one position internally during the forward pass.
    result["labels"] = result["input_ids"].copy()
    return result

train_ds = train_ds.map(
    group_texts,
    batched=True,
    batch_size=1024,
    num_proc=4
)

print(train_ds[0].keys())

Map (num_proc=4):   0%|          | 0/1212450 [00:00<?, ? examples/s]

dict_keys(['input_ids', 'attention_mask', 'labels'])


In [ ]:
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, 
    mlm=False
)

training_args = TrainingArguments(
    output_dir="./gpt2-trained-from-scratch",
    optim="adamw_torch_fused",
    learning_rate=2e-5,
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    per_device_train_batch_size=32,
    per_device_eval_batch_size=4,
    bf16=True,
    tf32=True,
    num_train_epochs=10,
    gradient_checkpointing=True,
    save_strategy="steps",
    logging_steps=1000,
    save_steps=10000
)

trainer = Trainer(
    model=model, 
    args=training_args,
    train_dataset=train_ds,
    data_collator=data_collator,
)

trainer.train()

/home/mardon/Documents/ai-homework/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  queued_call()
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


KeyboardInterrupt: 

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
trainer.push_to_hub(f"gpt2-{DATASET_NAME.split('/')[-1]}")

In [ ]:
trainer.save_model(f"./model/gpt2-{DATASET_NAME.split('/')[-1]}")
tokenizer.save_pretrained(f"./model/gpt2-{DATASET_NAME.split('/')[-1]}")

In [ ]:
def generate_text(query, model, tokenizer, max_new_tokens=50, device=None):
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    model.eval()
    model.to(device)
    
    inputs = tokenizer(query, return_tensors="pt").to(device)
    
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,      # How many new tokens to generate
            do_sample=True,                     # Enable sampling (adds randomness/creativity)
            temperature=0.8,                    # Higher = more random; Lower = more predictable
            top_k=50,                           # Limits sampling to the top 50 most likely tokens
            pad_token_id=tokenizer.eos_token_id # Prevents warnings since GPT-2 lacks a native pad token
        )
        
    generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    
    return generated_text

generate_text("salom", model, tokenizer, max_new_tokens=100)